In [1]:
# =========================
# Standard library
# =========================
import contextlib
import io
import json
import operator
import os
import re
import subprocess
import traceback
import warnings
from pathlib import Path
from typing import Annotated, Any, Literal, Optional, TypedDict


# =========================
# Third-party libraries
# =========================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError


# =========================
# LangChain / LangGraph
# =========================
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


# =========================
# Global settings
# =========================
pd.set_option("display.max_columns", 100)
warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
load_dotenv()

True

In [3]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API Key: {API_KEY is not None}")

API Key: True


In [4]:
model = "deepseek/deepseek-v4-flash"
# model = 'google/gemma-4-26b-a4b-it'
model = 'google/gemini-3-flash-preview'
# model = 'openai/gpt-5.4-mini'

llm_orchestrator = ChatOpenAI(
    model=model,
    api_key=API_KEY,
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0,
    max_tokens=3000
)

In [5]:
class AgentState(TypedDict, total=False):
    run_id: str | None

    dataset_path: str | None
    prepared_dataset_path: str | None

    target_column: str | None
    task_type: Literal["classification", "regression", "auto"]

    schema: dict[str, list[str]]
    data_summary: dict[str, Any]

    prep_summary: dict[str, Any]
    data_preparation: dict[str, Any]

    modeling_summary: dict[str, Any]

    next_input: dict[str, Any]

    artifacts: dict[str, str | None]
    logs: Annotated[list[dict[str, Any]], operator.add]

    current_step: str | None
    next_action: str | None
    orchestration_decision: dict[str, Any]
    orchestration_history: Annotated[list[dict[str, Any]], operator.add]

    previous_best_model_path: str | None
    previous_metrics_path: str | None

    retry_count: int
    max_retries: int
    status: str
    error: str | None

In [6]:
ORCHESTRATOR_SYSTEM_PROMPT = """
Ты — LLM-оркестратор ML workflow.

Твоя роль:
- анализировать текущий AgentState
- решать, что должно произойти дальше
- выбирать, какого специализированного агента нужно вызвать
- кратко объяснять причину решения
- обрабатывать ошибки, повторные попытки и завершение workflow
- использовать доступные инструменты только когда нужна безопасная проверка файловой системы
- использовать dataset_path, найденный на этапе инициализации
- использовать previous_best_model_path и previous_metrics_path, найденные на этапе инициализации, если они есть

Ты НЕ обучаешь модели напрямую.
Ты НЕ делаешь предобработку данных напрямую.
Ты НЕ создаёшь отчёты напрямую.
Ты только управляешь workflow.

Инициализация перед запуском оркестратора:
До запуска orchestrator_agent граф уже должен выполнить:
1. dataset_finder
2. previous_run_finder
3. join_init

Этап инициализации отвечает за:
- поиск пути к входному CSV-датасету
- игнорирование CSV-файлов внутри ./artifacts
- игнорирование predictions.csv, metrics.csv и временных output-файлов
- поиск предыдущих ML-артефактов, если они существуют
- первичную проверку текущих modeling-артефактов:
  - artifacts/modeling/final_metrics.json
  - artifacts/modeling/best_model.joblib
  - artifacts/modeling/modeling_summary.json

Оркестратор не должен напрямую вызывать dataset_finder или previous_run_finder.
Если после инициализации dataset_path отсутствует, выбери fail.

Доступные инструменты:
- bash_tool: используй только для безопасной проверки файловой системы, например поиска CSV-файлов, просмотра папок с артефактами или чтения JSON-метаданных.

Доступные решения:
- run_data_description
- run_data_preparation
- run_modeling
- run_reporting
- retry_current_step
- finish
- fail

Специализированные агенты:
- data_description_agent
- data_preparation_agent
- modeling_agent
- reporting_agent

Текущий workflow:
0. Инициализация уже выполнена до оркестратора:
   - dataset_finder
   - previous_run_finder
   - join_init
1. data_description_agent
2. data_preparation_agent
3. modeling_agent
4. reporting_agent

Правило про предыдущий запуск:
- Наличие предыдущего запуска НЕ означает, что текущий workflow уже выполнен.
- Не пропускай текущее моделирование только потому, что предыдущий запуск существует.
- Не выбирай finish только потому, что предыдущий запуск существует.
- previous_best_model_path и previous_metrics_path можно использовать только как контекст для сравнения, отчёта или fallback-комментариев.
- Текущий modeling_agent считается выполненным только по правилам completion ниже, основанным на текущих modeling_summary и artifacts.

Шаг считается выполненным ТОЛЬКО по следующим точным правилам:

1. data_description_agent выполнен, если:
   - data_description содержит ключ "status"
   - data_description["status"] == "success"
   ИЛИ выполнены все условия:
   - data_summary содержит ключ "n_rows"
   - data_summary содержит ключ "n_columns"
   - schema содержит ключ "numeric"
   - schema содержит ключ "categorical"
   - schema содержит ключ "text"
   - schema содержит ключ "datetime"

2. data_preparation_agent выполнен, если:
   - data_preparation содержит ключ "status"
   - data_preparation["status"] == "success"
   - artifacts содержит ключ "prepared_dataset_path"
   - artifacts["prepared_dataset_path"] != null

3. modeling_agent выполнен, если:
   - modeling_summary содержит ключ "status"
   - modeling_summary["status"] == "success"
   - modeling_summary содержит ключ "best_model_name"
   - modeling_summary["best_model_name"] != null
   - artifacts содержит ключ "best_model_path"
   - artifacts["best_model_path"] != null

4. reporting_agent выполнен, если:
   - final_report содержит ключ "status"
   - final_report["status"] == "success"
   ИЛИ выполнены все условия:
   - artifacts содержит ключ "final_report_path"
   - artifacts["final_report_path"] != null

Порядок принятия решения:
1. Если dataset_path отсутствует или равен null, выбери fail.
2. Если error не равен null и retry_count < max_retries, выбери retry_current_step.
3. Если error не равен null и retry_count >= max_retries, выбери fail.
4. Если data_description_agent не выполнен, выбери run_data_description.
5. Иначе если data_preparation_agent не выполнен, выбери run_data_preparation.
6. Иначе если modeling_agent не выполнен, выбери run_modeling.
7. Иначе если reporting_agent не выполнен, выбери run_reporting.
8. Иначе выбери finish.

Строгие правила:
- Не придумывай дополнительные требования к завершению шагов.
- Не говори, что шаг "not fully populated", если точные completion-keys существуют.
- Не выбирай run_data_description, если data_description["status"] равно "success" или если data_summary и schema содержат completion-ключи.
- Не выбирай run_data_preparation, если data_preparation["status"] равно "success" и prepared_dataset_path существует.
- Не выбирай run_modeling до завершения data_preparation_agent.
- Modeling можно запускать только после появления prepared_dataset_path.
- Не выбирай run_reporting до завершения modeling_agent.
- Не выбирай retry_current_step, если error равен null.
- Не выбирай fail, если error равен null, кроме случая, когда отсутствует dataset_path.
- Никогда не используй устаревшие решения: run_tabular_preparation, run_text_preparation, merge_features, run_improvement, use_fallback.
- Никогда не используй устаревших агентов: tabular_preparation_agent, text_preparation_agent, merge_features, improvement_agent.
- Используй bash_tool только для безопасной проверки файловой системы.
- Не удаляй, не перемещай, не перезаписывай и не изменяй файлы через bash_tool.
- При поиске датасета предпочитай CSV-файлы внутри ./data.
- Игнорируй CSV-файлы внутри ./artifacts.
- Игнорируй predictions.csv, metrics.csv и временные output CSV-файлы.
- Когда требуется структурированное решение, возвращай только валидный JSON.

Обязательный JSON-формат:
{
  "decision": "одно из доступных решений",
  "reason": "краткая причина решения",
  "selected_agent": "один из доступных специализированных агентов или null",
  "risk_notes": []
}
"""

In [7]:
initial_state = {
    "run_id": None,

    "dataset_path": None,
    "prepared_dataset_path": None,

    "target_column": "price",
    "task_type": "auto",

    "schema": {},
    "data_summary": {},

    "prep_summary": {},
    "data_preparation": {},

    "modeling_summary": {},

    "next_input": {},

    "artifacts": {},
    "logs": [],

    "current_step": None,
    "next_action": None,
    "orchestration_decision": {},
    "orchestration_history": [],

    "previous_run": {},
    "previous_best_model": {},

    "retry_count": 0,
    "max_retries": 1,
    "status": "running",
    "error": None,
}

In [8]:
@tool
def bash_tool(command: str) -> dict[str, Any]:
    """
    Выполняет bash-команду и возвращает stdout, stderr и return code.

    Используй этот инструмент только для безопасной проверки файловой системы:
    - найти CSV-файлы
    - посмотреть список файлов и папок
    - прочитать JSON-файлы
    - проверить существование файла

    Не используй этот инструмент для удаления, перемещения,
    перезаписи или изменения файлов
    """

    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30,
        )

        return {
            "status": "success" if result.returncode == 0 else "failed",
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
            "returncode": result.returncode,
        }

    except Exception as e:
        return {
            "status": "failed",
            "stdout": "",
            "stderr": str(e),
            "returncode": -1,
        }

In [9]:
tools = [bash_tool]

orchestrator_tool_agent = create_agent(
    model=llm_orchestrator,
    tools=tools,
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
)

In [10]:
def extract_csv_path(text: str) -> str | None:
    match = re.search(r"([./\w\-/]+\.csv)", text)
    if match:
        return match.group(1)
    return None


def dataset_finder_node(state: AgentState) -> AgentState:
    print("=== ЗАПУСК ПОИСКА ДАТАСЕТА ===")

    result = orchestrator_tool_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Найди основной CSV-датасет в текущей папке проекта"
                    "При необходимости используй bash_tool"
                    "Предпочитай файлы внутри ./data"
                    "Игнорируй файлы внутри ./artifacts"
                    "Игнорируй predictions.csv, metrics.csv и временные output-файлы"
                    "Верни только путь к датасету обычным текстом "
                    "Не добавляй объяснения."
                )
            }
        ]
    })

    agent_response = result["messages"][-1].content.strip()

    print("Ответ dataset_finder:", agent_response)

    dataset_path = extract_csv_path(agent_response)

    if not dataset_path:
        print("Поиск датасета завершился ошибкой")

        return {
            "error": f"Путь к датасету не найден. {agent_response}",
            "status": "failed",
            "logs": [{
                "agent": "dataset_finder",
                "status": "failed",
                "summary": "Не удалось найтиь датасет",
                "decisions": {
                    "agent_response": agent_response
                },
                "artifacts": {},
                "next_input": {},
                "reason": "Датасет не найден",
            }],
        }

    print(f"Поиск датасета завершён. Путь к датасету: {dataset_path}")

    return {
        "dataset_path": dataset_path,
        "status": "running",
        "error": None,
        "logs": [{
            "agent": "dataset_finder",
            "status": "success",
            "summary": f"Датасет найден: {dataset_path}",
            "decisions": {
                "agent_response": agent_response,
            },
            "artifacts": {},
            "next_input": {
                "dataset_path": dataset_path,
            },
            "reason": None,
        }],
    }

In [11]:
def extract_json_safe(text: str) -> dict | None:
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    return None


def previous_run_finder_node(state: AgentState) -> AgentState:
    print("=== ПОИСК ПРЕДЫДУЩЕГО ЗАПУСКА ===")

    result = orchestrator_tool_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Найди последний предыдущий ML-запуск внутри папки artifacts. "
                    "При необходимости используй bash_tool. "
                    "Ищи артефакты, которые могут содержать metrics.json, final_metrics.json, "
                    "best_model.pkl или best_model.joblib. "
                    "Сначала проверь текущие modeling-artifacts: "
                    "artifacts/modeling/final_metrics.json, "
                    "artifacts/modeling/best_model.joblib, "
                    "artifacts/modeling/modeling_summary.json. "
                    "Верни только валидный JSON, без markdown и без объяснений. "
                    "Формат JSON:\n"
                    "{\n"
                    '  "metrics_path": "path or null",\n'
                    '  "best_model_path": "path or null"\n'
                    "}"
                )
            }
        ]
    })

    agent_response = result["messages"][-1].content.strip()
    print("previous_run_finder:", agent_response)

    parsed = extract_json_safe(agent_response)

    if parsed is None:
        parsed = {
            "metrics_path": None,
            "best_model_path": None,
        }

    metrics_path = parsed.get("metrics_path")
    best_model_path = parsed.get("best_model_path")

    if metrics_path in ["null", "None", ""]:
        metrics_path = None

    if best_model_path in ["null", "None", ""]:
        best_model_path = None

    previous_run_found = bool(metrics_path or best_model_path)

    if not previous_run_found:
        log_record = {
            "agent": "previous_run_finder",
            "status": "success",
            "skipped": True,
            "summary": "Предыдущая лучшая модель или метрики не найдены",
            "decisions": {
                "agent_response": agent_response,
                "previous_best_model_path": None,
                "previous_metrics_path": None,
            },
            "artifacts": {},
            "reason": "Пути к предыдущей модели или метрикам не найдены.",
        }

        return {
            "previous_best_model_path": None,
            "previous_metrics_path": None,
            "logs": [log_record],
        }

    log_record = {
        "agent": "previous_run_finder",
        "status": "success",
        "skipped": False,
        "summary": "Предыдущая лучшая модель или метрики найдены.",
        "decisions": {
            "agent_response": agent_response,
            "previous_best_model_path": best_model_path,
            "previous_metrics_path": metrics_path,
        },
        "artifacts": {},
        "reason": None,
    }

    return {
        "previous_best_model_path": best_model_path,
        "previous_metrics_path": metrics_path,
        "logs": [log_record],
    }

In [12]:
def join_init_node(state: AgentState) -> AgentState:
    print("=== ОБЪЕДИНЕНИЕ ===")

    dataset_path = state.get("dataset_path")
    previous_best_model_path = state.get("previous_best_model_path")
    previous_metrics_path = state.get("previous_metrics_path")

    log_record = {
        "agent": "join_init",
        "status": "success",
        "summary": "Инициализация завершена: поиск датасета и поиск предыдущих артефактов выполнены",
        "decisions": {
            "dataset_path": dataset_path,
            "previous_best_model_path": previous_best_model_path,
            "previous_metrics_path": previous_metrics_path,
        },
        "artifacts": {},
        "reason": None,
    }

    return {
        "current_step": "join_init",
        "logs": [log_record],
    }

In [13]:
class OrchestratorDecision(BaseModel):
    decision: Literal[
        "run_data_description",
        "run_data_preparation",
        "run_modeling",
        "run_improvement",
        "run_reporting",
        "retry_current_step",
        "finish",
        "fail",
    ] = Field(
        description="Следующее действие, которое выбрал orchestrator"
    )

    reason: str = Field(
        description="Краткое объяснение, почему выбрано именно это действие"
    )

    selected_agent: Optional[
        Literal[
            "data_description_agent",
            "data_preparation_agent",
            "modeling_agent",
            "reporting_agent",
        ]
    ] = Field(
        default=None,
        description="Название агента, которого нужно вызвать следующим"
    )

    risk_notes: list[str] = Field(
        default_factory=list,
        description="Риски или важные замечания по текущему шагу"
    )

In [16]:
def call_orchestrator_llm_orchestrator(state: dict) -> OrchestratorDecision:
    """
    Отправляет текущее состояние AgentState в LLM-оркестратор
    и возвращает провалидированный объект OrchestratorDecision.
    """

    state_for_llm_orchestrator = {
        "run_id": state.get("run_id"),
        "dataset_path": state.get("dataset_path"),
        "target_column": state.get("target_column"),
        "task_type": state.get("task_type"),

        "schema": state.get("schema", {}),
        "data_summary": state.get("data_summary", {}),
        "prep_summary": state.get("prep_summary", {}),
        "data_preparation": state.get("data_preparation", {}),
        "modeling_summary": state.get("modeling_summary", {}),
        "final_report": state.get("final_report", {}),
        "artifacts": state.get("artifacts", {}),

        "previous_best_model_path": state.get("previous_best_model_path"),
        "previous_metrics_path": state.get("previous_metrics_path"),

        "available_keys": {
            "schema_keys": list(state.get("schema", {}).keys()),
            "data_summary_keys": list(state.get("data_summary", {}).keys()),
            "prep_summary_keys": list(state.get("prep_summary", {}).keys()),
            "data_preparation_keys": list(state.get("data_preparation", {}).keys()),
            "modeling_summary_keys": list(state.get("modeling_summary", {}).keys()),
            "final_report_keys": list(state.get("final_report", {}).keys()),
            "artifact_keys": list(state.get("artifacts", {}).keys()),
        },

        "logs_count": len(state.get("logs", [])),
        "current_step": state.get("current_step"),
        "next_action": state.get("next_action"),
        "orchestration_history": state.get("orchestration_history", [])[-5:],

        "retry_count": state.get("retry_count", 0),
        "max_retries": state.get("max_retries", 1),
        "status": state.get("status"),
        "error": state.get("error"),
    }

    response = llm_orchestrator.invoke(
        [
            SystemMessage(content=ORCHESTRATOR_SYSTEM_PROMPT),
            HumanMessage(
                content=(
                    "Проанализируй текущий AgentState и выбери следующее действие.\n\n"
                    "Верни ТОЛЬКО валидный JSON. "
                    "Не используй markdown. "
                    "Не добавляй объяснения вне JSON.\n"
                    "Формат JSON:\n"
                    "{\n"
                    '  "decision": "run_data_description",\n'
                    '  "reason": "краткая причина",\n'
                    '  "selected_agent": "data_description_agent",\n'
                    '  "risk_notes": []\n'
                    "}\n\n"
                    f"AgentState:\n{json.dumps(state_for_llm_orchestrator, ensure_ascii=False, indent=2)}"
                )
            ),
        ]
    )

    raw_text = response.content
    parsed = extract_json_safe(raw_text)

    if parsed is None:
        raise ValueError(
            "LLM-оркестратор вернул ответ, который не удалось распарсить как JSON.\n"
            f"Сырой ответ: {raw_text}"
        )

    try:
        return OrchestratorDecision(**parsed)
    except ValidationError as e:
        raise ValueError(
            "Неправильный ответ\n"
            f"Ошибка валидации: {e}\n"
            f"Ответ: {raw_text}"
        )

In [15]:
def orchestrator_agent_node(state: AgentState) -> AgentState:
    decision = call_orchestrator_llm_orchestrator(state)
    decision_dict = decision.model_dump()

    log_record = {
        "agent": "orchestrator_agent",
        "status": "success",
        "summary": decision.reason,
        "decisions": decision_dict,
        "artifacts": {},
        "reason": None,
    }

    return {
        "current_step": "orchestrator_agent",
        "next_action": decision.decision,
        "orchestration_decision": decision_dict,
        "orchestration_history": [decision_dict],
        "logs": [log_record],
    }